# Gradient Cobra

## Combine Classifier

In [9]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(
    f"Training set: {X_train.shape[0]} samples, {X_train.shape[1]} features\n"
    f"Test set: {X_test.shape[0]} samples, {X_test.shape[1]} features"
)

Training set: 455 samples, 30 features
Test set: 114 samples, 30 features


In [10]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from cobra.combine_classifier import CombineClassifier

model = CombineClassifier(
    splitter="holdout",
    distance="hamming",
    kernel="indicator",
    aggregator="majority_vote",
    random_state=42
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

In [11]:
preds = model.predict(X_test)
accuracy = (preds == y_test).mean()
print(f"Test set accuracy: {accuracy:.4f}")

Test set accuracy: 0.9649


In [12]:
from sklearn.metrics import accuracy_score

estimators = [
    "decision_tree",
    "logistic_regression",
    "random_forest",
    "gradient_boosting",
    "knn",
    "svm"
]

results = []

for estimator in estimators:
    model = CombineClassifier(
        estimators=[estimator],
        splitter="holdout",
        distance="hamming",
        kernel="indicator",
        aggregator="majority_vote",
        random_state=42
    )

    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    acc = accuracy_score(y_test, preds)

    results.append((estimator, acc))
    print(f"{estimator:20s} accuracy: {acc:.4f}")

KeyError: "'decision_tree' is not registered in EstimatorFactory. Available: ['desicion_tree', 'dummy_mean', 'gradient_boosting', 'knn', 'lasso', 'linear', 'logistic_regression', 'mean_regressor', 'random_forest', 'ridge', 'svm']"

In [5]:
from cobra.core.estimators.base import BaseEstimator, EstimatorFactory
EstimatorFactory.available()

['desicion_tree',
 'dummy_mean',
 'gradient_boosting',
 'knn',
 'lasso',
 'linear',
 'logistic_regression',
 'mean_regressor',
 'random_forest',
 'ridge',
 'svm']

# GradientCobra

In [6]:
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR

from cobra.gradientcobra import GradientCOBRA


# ======================
# DATA
# ======================
X, y = fetch_california_housing(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)


# ======================
# SINGLE MODELS
# ======================
single_models = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "RandomForest": RandomForestRegressor(n_estimators=300, random_state=42),
    "SVR": SVR(C=5.0, epsilon=0.1)
}

results = {}


for name, model in single_models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    mse = mean_squared_error(y_test, pred)
    results[name] = mse


# ======================
# GRADIENT COBRA
# ======================
cobra = GradientCOBRA(
    kernel="rbf",
    distance="euclidean",
    aggregator="weighted_mean",
    optimizer="grid",
    bandwidth_grid=np.linspace(0.1, 5.0, 15),
    optimizer_params={"verbose": True},
    random_state=42
)

cobra.fit(X_train, y_train)
cobra_pred = cobra.predict(X_test)

cobra_mse = mean_squared_error(y_test, cobra_pred)
results["GradientCOBRA"] = cobra_mse


# ======================
# RESULTS
# ======================
print("\n===== MSE COMPARISON (lower is better) =====\n")

for name, score in sorted(results.items(), key=lambda x: x[1]):
    print(f"{name:20s} : {score:.4f}")


===== MSE COMPARISON (lower is better) =====

RandomForest         : 0.2537
GradientCOBRA        : 0.2936
Ridge                : 0.5558
LinearRegression     : 0.5559
SVR                  : 1.2197


# MixCobra

In [7]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error

from cobra.mixcobra import MixCOBRARegressor

# ======================
# SINGLE MODELS
# ======================
single_models = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "RandomForest": RandomForestRegressor(n_estimators=300, random_state=42),
    "SVR": SVR(C=5.0, epsilon=0.1)
}

results = {}

# train single models
for name, model in single_models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    results[name] = mean_squared_error(y_test, pred)

# ======================
# MIXCOBRA
# ======================
mix = MixCOBRARegressor(
    kernel="rbf",
    distance="euclidean",
    aggregator="weighted_mean",
    alpha_grid=np.linspace(0.1, 3.0, 10),
    beta_grid=np.linspace(0.1, 3.0, 10),
    random_state=42
)

mix.fit(X_train, y_train)
mix_pred = mix.predict(X_test)

results["MixCOBRA"] = mean_squared_error(y_test, mix_pred)

# ======================
# RESULTS
# ======================
print("\n===== MSE COMPARISON =====\n")

for name, score in sorted(results.items(), key=lambda x: x[1]):
    print(f"{name:20s} : {score:.4f}")


===== MSE COMPARISON =====

RandomForest         : 0.2537
MixCOBRA             : 0.4114
Ridge                : 0.5558
LinearRegression     : 0.5559
SVR                  : 1.2197
